# HEP Multiagent Demo

This notebook demonstrates how to use the multi-agent framework for scientific research queries.

## Prerequisites
- Python 3.10+
- LaTeX distribution for PDF reports ([BasicTeX](https://www.tug.org/mactex/morepackages.html))
- Environment variables configured (see below)

In [11]:
# Development mode
!pip install -e ".[dev]"

# For production:
# !pip install -q --force-reinstall git+https://github.com/HEP-KE/HEP-multiagent.git


Obtaining file:///data/a/cpac/nramachandra/Projects/AmSC/HEP-multiagent
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for hep-multiagent (pyproject.toml) ... done
  Created wheel for hep-multiagent: filename=hep_multiagent-0.1.0-0.editable-py3-none-any.whl size=1543 sha256=c0c9f76d9b423012aa25906a9a1c8cc403c6a8a2fba01ea812250ada23d3bc32
  Stored in directory: /tmp/pip-ephem-wheel-cache-yeif_5v4/wheels/5e/5b/63/a54bf26a25740dabab95611287a6f712d6bd6f06f46d542b67
Successfully built hep-multiagent
  Attempting uninstall: hep-multiagent
    Found existing installation: hep-multiagent 0.1.0
    Uninstalling hep-multiagent-0.1.0:
      Successfully uninstalled hep-multiagent-0.1.0


In [12]:
!pip install -q --force-reinstall git+https://github.com/HEP-KE/mcp-ke.git
#install last since it needs mcp<1.23.0 but 1.26.0 got installed
!pip install -q --force-reinstall git+https://github.com/HEP-KE/kb-mcp.git

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.63.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.2 which is incompatible.
opencosmo 1.0.4 requires numpy<2.4,>=2.0, but you have numpy 2.4.2 which is incompatible.
kb-mcp 0.1.0 requires mcp<1.23.0, but you have mcp 1.26.0 which is incompatible.
Username for 'https://github.com': ERROR: Operation cancelled by user
^C


### Set up for KB MCP

Set up paths and choose which papers to download.

In [22]:
import os
import subprocess
import sys
from pathlib import Path

# Paths (stores data in current directory)
DATA_DIR = Path.cwd() / "data"
DB_PATH = DATA_DIR / "kb.db"
PAPERS_DIR = DATA_DIR / "papers"

# Papers to download (arXiv ID, title)
PAPERS = [
    ("1807.06209", "Planck 2018 cosmological parameters"),
    ("2007.08991", "eBOSS cosmological results"),
    ("1502.01589", "Planck 2015 cosmological results"),
]

print(f"Database: {DB_PATH}")
print(f"Papers: {PAPERS_DIR}")

Database: /data/a/cpac/nramachandra/Projects/AmSC/HEP-multiagent/data/kb.db
Papers: /data/a/cpac/nramachandra/Projects/AmSC/HEP-multiagent/data/papers


Initialize an empty SQLite database with the kb-mcp schema.

In [23]:
from kb_mcp.kb.db_models import Base
from sqlalchemy import create_engine

# Create directories
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
PAPERS_DIR.mkdir(parents=True, exist_ok=True)

# Create database
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.create_all(engine)

print(f"Created database: {DB_PATH}")

Created database: /data/a/cpac/nramachandra/Projects/AmSC/HEP-multiagent/data/kb.db


Fetch PDFs from arXiv and extract text.

In [24]:
from hep_multiagent.features.arxiv_fetch import download_full_text

for arxiv_id, title in PAPERS:
    txt_path = PAPERS_DIR / f"{arxiv_id}.txt"
    if txt_path.exists():
        print(f"{arxiv_id} - already downloaded")
    else:
        print(f"[downloading] {arxiv_id}: {title}")
        download_full_text(arxiv_id, str(PAPERS_DIR))

print(f"\nDownloaded {len(list(PAPERS_DIR.glob('*.txt')))} papers")

1807.06209 - already downloaded
2007.08991 - already downloaded
1502.01589 - already downloaded

Downloaded 3 papers


Ingest the downloaded papers into kb-mcp.

In [25]:
# NOTE: Embeddings required for kb_search to work with SQLite.
# Using --no-embed causes kb_search to crash (KeyError: 'total_results')
# because SQLite doesn't support full-text search.

os.environ["SQLITE_DB_PATH"] = str(DB_PATH)

for arxiv_id, _ in PAPERS:
    txt_path = PAPERS_DIR / f"{arxiv_id}.txt"
    if txt_path.exists():
        print(f"[ingesting] {arxiv_id}")
        
        subprocess.run(
            [sys.executable, "-m", "kb_mcp.kb.cli", "ingest", str(txt_path),
             "--source-id", "arxiv", "--no-summary", "--batch"],
            capture_output=True
        )

print("\nDone! Checking database...")
result = subprocess.run([sys.executable, "-m", "kb_mcp.kb.cli", "stats"], capture_output=True, text=True)
print(result.stdout)

[ingesting] 1807.06209
[ingesting] 2007.08991
[ingesting] 1502.01589

Done! Checking database...
Knowledge Base Statistics
Total documents: 3
Total sources: 1

Documents by source:
  arxiv: 3



### Set up HEP Multiagent

Set `ARGO_USER` in `.env` or pass env vars directly in server config.

In [26]:
import os
from datetime import datetime
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from hep_multiagent import Agent

load_dotenv(".env")

llm = ChatOpenAI(
    model="claudesonnet4",
    base_url="https://apps-dev.inside.anl.gov/argoapi/v1",
    api_key=os.environ.get("ARGO_USER", "")
)

## Initialize Agent

Create an agent with an LLM and MCP servers. Servers are auto-installed from URL at init.

In [ ]:
from hep_multiagent import Agent

agent = await Agent(
    llm=llm,
    mcp_servers=[
    {
        "url": "https://github.com/HEP-KE/kb-mcp.git",
        "name": "kb-server-stdio",
        "env": {"SQLITE_DB_PATH": str(DB_PATH)},
    },
    {
        "url": "https://github.com/HEP-KE/mcp-ke.git"
    },
    ],
    approval=False,
)

for tool in agent.tools:
    print(f"  - {tool.name}")

RuntimeError: pip install git+https://github.com/HEP-KE/kb-mcp.git failed:
  [1;31merror[0m: [1msubprocess-exited-with-error[0m
  
  [31m×[0m [32mgit clone --[0m[32mfilter[0m[32m=[0m[32mblob[0m[32m:none --quiet [0m[4;32mhttps://github.com/HEP-KE/kb-mcp.git[0m[32m [0m[32m/tmp/[0m[32mpip-req-build-ykm010e4[0m did not run successfully.
  [31m│[0m exit code: [1;36m128[0m
  [31m╰─>[0m [31m[1 lines of output][0m
  [31m   [0m fatal: could not read Username for 'https://github.com': No such device or address
  [31m   [0m [31m[end of output][0m
  
  [1;35mnote[0m: This error originates from a subprocess, and is likely not a problem with pip.
[31mERROR: Failed to build 'git+https://github.com/HEP-KE/kb-mcp.git' when git clone --filter=blob:none --quiet https://github.com/hep-ke/kb-mcp.git /tmp/pip-req-build-ykm010e4[0m[31m
[0m

In [10]:
agent.print_tools()

NameError: name 'agent' is not defined

Simple test question

In [19]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = f"./output_hepke_arxiv_demo_{timestamp}"

result = await agent.run(
    query = "Search arxiv for 5 papers on dark matter detection, Briefly provide information on the most recent method. then create a bar chart showing the publication year distribution of the papers found.",
    output_dir=OUTPUT_DIR,
)

NameError: name 'agent' is not defined

result = await agent.run(
    query = """
Using the observational data from eBOSS DR14 Lyman-alpha forest, 
compare the linear P(k) values for LCDM, LCDM with massive neutrinos (Emv=0.10 eV), and dark 
energy model with equation of state parameter w0=-0.9.

Create visualizations showing:
1. The power spectra comparison with observational data
2. The suppression ratios relative to LCDM

Comment on how close the P(k) values are and analyze the power spectrum suppression compared to LCDM.
""",
    output_dir="./output_kb_hep_mcp"
)

print(result)

In [20]:
mcmc_query = """Run everything via MCP tools in mcp-ke. 
Absolutely do not use your own code (no writing new python codes, existing tools should be used). 
Strictly no fake/synthetic/placeholder data. 
If something doesn't work, do not try to find a workaround or shortcuts that need writing realistic estimations. 

(1) First, load observational data from eBOSS (see: /data/a/cpac/nramachandra/Projects/AmSC/mcp-ke/data/DR14_pm3d_19kbins.txt).
(2) Then compare the P(k) with wCDM, ΛCDM + Massive Neutrinos and ΛCDM (use any set of parameters you need). Plot them all.
(3) Finally, run a full posterior analysis using MCMC? Do this for 3 parameters (sigma8, Σmν, N_species) of the 
ΛCDM + massive neutrinos model. Use priors from Planck. 
Go with a small run. 
(4) Show me the final posterior distribution plot from GetDist and the best-fit estimates.
"""

In [21]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = f"./output_hepke_arxiv_demo_{timestamp}"

result = await agent.run(
    query = mcmc_query,
    output_dir=OUTPUT_DIR,
)

NameError: name 'agent' is not defined